In [ ]:
# Install required libraries
!pip install wfdb tensorflow numpy matplotlib scikit-learn

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0"
files = os.listdir(dataset_path)
print(f"Total files: {len(files)}")
files[:10]

Total files: 207


['100.atr',
 '100.hea',
 '100.dat',
 '101.dat',
 '101.atr',
 '101.hea',
 '102.dat',
 '102.hea',
 '103.dat',
 '103.atr']

In [ ]:
# import wfdb
# import numpy as np
# from collections import Counter

# # Heartbeat window: 180 samples before and after each R-peak
# WINDOW_SIZE = 180

# # MIT-BIH record_names
# record_names = [f.replace('.hea', '') for f in files if f.endswith('.hea')]
# print(f"Total record: {len(record_names)}")

# # Label mapping; N is normal, everything else is abnormal
# # Full AAMI standard grouping
# NORMAL_BEATS = ['N', 'L', 'R', 'e', 'j']  # Normal and bundle branch blocks
# ABNORMAL_BEATS = ['A', 'a', 'J', 'S', 'V', 'E', 'F']  # Supraventricular + Ventricular

# X = []  # ECG segments
# y = []  # Labels

# for record in record_names:
#     try:
#         # Load the record
#         sig, fields = wfdb.rdsamp(os.path.join(dataset_path, record))
#         ann = wfdb.rdann(os.path.join(dataset_path, record), 'atr')

#         # Use Lead 0 (MLII) only
#         ecg = sig[:, 0]

#         # Extract beats
#         for i, (sample, symbol) in enumerate(zip(ann.sample, ann.symbol)):
#             if symbol in NORMAL_BEATS:
#                 label = 0
#             elif symbol in ABNORMAL_BEATS:
#                 label = 1
#             else:
#                 continue
#             # extract WINDOW around R-peak
#             start = sample - WINDOW_SIZE
#             end = sample + WINDOW_SIZE

#             if start >= 0 and end <= len(ecg):
#                 segment = ecg[start:end]
#                 X.append(segment)
#                 y.append(label)
#     except Exception as e:
#         print(f"Skipping {record}: {e}")
# X = np.array(X)
# y = np.array(y)

# print(f"\nTotal beats extracted: {len(X)}")
# print(f"Shape: {X.shape}")
# print(f"Label distribution: {Counter(y)}")
# print(f"Normal: {sum(y==0)}, Abnormal: {sum(y==1)}")

In [ ]:
import wfdb
import numpy as np
from collections import Counter

WINDOW_SIZE = 180

# Standard patient-wise split (AAMI standard)
TRAIN_RECORDS = [
    '100', '101', '103', '105', '106', '107', '108', '109',
    '111', '112', '113', '114', '115', '116', '117', '118',
    '119', '121', '122', '123', '124', '200', '201', '202',
    '203', '205', '207', '208', '209', '210', '212', '213',
    '214', '215', '217', '219', '220', '221', '222', '223',
    '228', '230', '231', '232', '233', '234'
]

TEST_RECORDS = [
    '102', '104', '111', '113', '117', '123', '124', '200',
    '202', '210', '212', '213', '214', '219', '221', '222',
    '228', '231', '232', '233', '234'
]

NORMAL_BEATS = ['N', 'L', 'R', 'e', 'j']
ABNORMAL_BEATS = ['A', 'a', 'J', 'S', 'V', 'E', 'F']

def extract_beats(record_list):
    X, y = [], []
    for record in record_list:
        try:
            sig, fields = wfdb.rdsamp(os.path.join(dataset_path, record))
            ann = wfdb.rdann(os.path.join(dataset_path, record), 'atr')
            ecg = sig[:, 0]
            for sample, symbol in zip(ann.sample, ann.symbol):
                if symbol in NORMAL_BEATS:
                    label = 0
                elif symbol in ABNORMAL_BEATS:
                    label = 1
                else:
                    continue
                start = sample - WINDOW_SIZE
                end = sample + WINDOW_SIZE
                if start >= 0 and end <= len(ecg):
                    X.append(ecg[start:end])
                    y.append(label)
        except Exception as e:
            print(f"Skipping {record}: {e}")
    return np.array(X), np.array(y)

print("Extracting training beats...")
X_train_raw, y_train_raw = extract_beats(TRAIN_RECORDS)
print(f"Train beats: {len(X_train_raw)}")
print(f"Train distribution: Normal={sum(y_train_raw==0)}, Abnormal={sum(y_train_raw==1)}")

print("\nExtracting test beats...")
X_test_raw, y_test_raw = extract_beats(TEST_RECORDS)
print(f"Test beats: {len(X_test_raw)}")
print(f"Test distribution: Normal={sum(y_test_raw==0)}, Abnormal={sum(y_test_raw==1)}")

Extracting training beats...
Train beats: 101132
Train distribution: Normal=90320, Abnormal=10812

Extracting test beats...
Test beats: 42784
Test distribution: Normal=37329, Abnormal=5455


In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.utils import resample

# # Normalise each beat to zero mean
# X_norm = (X - X.mean(axis=1, keepdims=True) / X.std(axis=1, keepdims=True) + 1e-8)

# # Handle class imbalance
# X_normal = X_norm[y == 0]
# X_abnormal = X_norm[y == 1]
# y_normal = y[y == 0]
# y_abnormal = y[y == 1]

# # Upsample abnormal to match normal
# X_abnormal_upsampled, y_abnormal_upsampled = resample(
#     X_abnormal, y_abnormal,
#     replace = True,
#     n_samples = len(X_normal),
#     random_state = 42
# )

# # Combine and reshuffle
# X_balanced = np.concatenate([X_normal, X_abnormal_upsampled])
# y_balanced = np.concatenate([y_normal, y_abnormal_upsampled])

# # Reshape for CNN input: (samples, timesteps, channels)
# X_balanced = X_balanced.reshape(-1, 360, 1)

# # Split into train, validation and test
# X_train, X_temp, y_train, y_temp = train_test_split(
#     X_balanced, y_balanced, test_size=0.3, random_state=42, stratify=y_balanced
# )

# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
# )

# print(f"Training samples: {len(X_train)}")
# print(f"Validation samples: {len(X_val)}")
# print(f"Test samples: {len(X_test)}")
# print(f"Input shape: {X_train.shape}")

In [ ]:
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

# STEP 1: Normalize
print("Normalizing...")
X_train_norm = (X_train_raw - X_train_raw.mean(axis=1, keepdims=True)) / (X_train_raw.std(axis=1, keepdims=True) + 1e-8)
X_test_norm = (X_test_raw - X_test_raw.mean(axis=1, keepdims=True)) / (X_test_raw.std(axis=1, keepdims=True) + 1e-8)

# STEP 2: Balance training set only (never touch test set)
print("Balancing training set...")
X_tr_normal = X_train_norm[y_train_raw == 0]
X_tr_abnormal = X_train_norm[y_train_raw == 1]
y_tr_normal = y_train_raw[y_train_raw == 0]
y_tr_abnormal = y_train_raw[y_train_raw == 1]

X_tr_abnormal_up, y_tr_abnormal_up = resample(
    X_tr_abnormal, y_tr_abnormal,
    replace=True,
    n_samples=len(X_tr_normal),
    random_state=42
)

X_train_balanced = np.vstack([X_tr_normal, X_tr_abnormal_up])
y_train_balanced = np.hstack([y_tr_normal, y_tr_abnormal_up])

print(f"Balanced train: Normal={sum(y_train_balanced==0)}, Abnormal={sum(y_train_balanced==1)}")

# STEP 3: Carve validation set from training set
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_balanced, y_train_balanced,
    test_size=0.15,
    random_state=42,
    stratify=y_train_balanced
)

# STEP 4: Reshape for CNN (samples, timesteps, channels)
X_train = X_train_final.reshape(-1, 360, 1)
y_train = y_train_final
X_val = X_val.reshape(-1, 360, 1)
X_test = X_test_norm.reshape(-1, 360, 1)
y_test = y_test_raw

print(f"\nFinal shapes:")
print(f"X_train: {X_train_final.shape}, y_train: {y_train_final.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}, y_test: {y_test.shape}")
print(f"\nTest distribution: Normal={sum(y_test==0)}, Abnormal={sum(y_test==1)}")

Normalizing...
Balancing training set...
Balanced train: Normal=90320, Abnormal=90320

Final shapes:
X_train: (153544, 360), y_train: (153544,)
X_val:   (27096, 360, 1), y_val: (27096,)
X_test:  (42784, 360, 1), y_test: (42784,)

Test distribution: Normal=37329, Abnormal=5455


In [ ]:
# # Split the dataset into 5 parts for intial run
# subset_size = len(X_train) // 5

# X_train_sub = X_train[:subset_size]
# y_train_sub = y_train[:subset_size]

# X_val_sub = X_val[:len(X_val) // 5]
# y_val_sub = y_val[:len(y_val) // 5]

# print(f"Trial training samples: {len(X_train_sub)}")
# print(f"Trial validation samples: {len(X_val_sub)}")
# print(f"Input shape: {X_train_sub.shape}")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Build a 1D lightweight model
def build_model(input_shape=(360, 1)):
    model = keras.Sequential([
        # Block 1
        layers.Conv1D(32, kernel_size=5, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        # Block 2
        layers.Conv1D(32, kernel_size=5, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        # Block 3
        layers.Conv1D(32, kernel_size=3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        layers.Dropout(0.2),

        # Classifier Head
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_model()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_6 (Conv1D)               │ (None, 356, 32)        │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 356, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_6 (MaxPooling1D)  │ (None, 178, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 178, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 174, 32)        │         5,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 174, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_7 (MaxPooling1D)  │ (None, 87, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 87, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 85, 32)         │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 85, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 42, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 42, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,009 (43.00 KB)

 Trainable params: 10,817 (42.25 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
def build_model_v2(input_shape=(360, 1)):
    inputs = keras.Input(shape=input_shape)

    # Block 1: broad pattern detection
    x = layers.Conv1D(16, kernel_size=5, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2: mid-level features
    x = layers.Conv1D(24, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)  # Only one dropout here

    # Block 3: fine local features
    x = layers.Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Classifier
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)  # One dropout before output
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_model_v2()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 360, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 360, 16)        │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 360, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 180, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 180, 24)        │         1,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 180, 24)        │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 90, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 90, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 90, 32)         │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 90, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 45, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,753 (22.47 KB)

 Trainable params: 5,609 (21.91 KB)

 Non-trainable params: 144 (576.00 B)


Total parameters: 5,753


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print("\nTrial training complete.")

Epoch 1/20
4799/4799 ━━━━━━━━━━━━━━━━━━━━ 185s 36ms/step - accuracy: 0.9046 - loss: 0.2532 - precision: 0.9231 - recall: 0.8828 - val_accuracy: 0.9431 - val_loss: 0.1620 - val_precision: 0.9605 - val_recall: 0.9242 - learning_rate: 0.0010
Epoch 2/20
4799/4799 ━━━━━━━━━━━━━━━━━━━━ 159s 33ms/step - accuracy: 0.9351 - loss: 0.1838 - precision: 0.9523 - recall: 0.9161 - val_accuracy: 0.9503 - val_loss: 0.1378 - val_precision: 0.9582 - val_recall: 0.9417 - learning_rate: 0.0010
Epoch 3/20
4799/4799 ━━━━━━━━━━━━━━━━━━━━ 153s 32ms/step - accuracy: 0.9417 - loss: 0.1647 - precision: 0.9583 - recall: 0.9236 - val_accuracy: 0.9527 - val_loss: 0.1364 - val_precision: 0.9739 - val_recall: 0.9304 - learning_rate: 0.0010
Epoch 4/20
4799/4799 ━━━━━━━━━━━━━━━━━━━━ 142s 30ms/step - accuracy: 0.9457 - loss: 0.1529 - precision: 0.9617 - recall: 0.9285 - val_accuracy: 0.9050 - val_loss: 0.2413 - val_precision: 0.8605 - val_recall: 0.9667 - learning_rate: 0.0010
Epoch 5/20
4799/4799 ━━━━━━━━━━━━━━━━━━━━ 13

In [ ]:
# Evaluate on full test set
print("Evaluating on test set...")
results = model.evaluate(X_test, y_test, verbose=1)

print(f"\nTest Accuracy:  {results[1]*100:.2f}%")
print(f"Test Precision: {results[2]*100:.2f}%")
print(f"Test Recall:    {results[3]*100:.2f}%")

Evaluating on test set...
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 15s 11ms/step - accuracy: 0.9800 - loss: 0.0694 - precision: 0.8881 - recall: 0.9650

Test Accuracy:  98.00%
Test Precision: 88.81%
Test Recall:    96.50%


In [ ]:
import time
import os
import numpy as np

# Convert to TFLite (float32, no quantiyation yet)
print(f"Converting to TFLite (float32)...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = '/content/drive/MyDrive/ecg_model_float32.tflite'

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
float32_size = os.path.getsize(tflite_path) / 1024
print(f"Float32 model size: {float32_size:.2f} KB")

# Convert to TFLite with INT8 quantization
print(f"/n Converting to TFLite (INT8 quanttized)...")

def representative_dataset():
    for i in range(0, 500, 1):
        yield [X_test[i:i+1].astype(np.float32)]

converter_q = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
converter_q.representative_dataset = representative_dataset
converter_q.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_q.inference_input_type = tf.float32
converter_q.inference_output_type = tf.float32

tflite_quant_model = converter_q.convert()

quant_path = '/content/drive/MyDrive/ecg_model_int8.tflite'
with open(quant_path, 'wb') as f:
    f.write(tflite_quant_model)

quant_size = os.path.getsize(quant_path) / 1024
print(f"INT8 quantized model size: {quant_size:.2f} KB")
print(f"Size reduction: {float32_size/quant_size:.2f}x smaller")

# Benchmark inference speed on CPU only
print("\nBenchmarking inference speed on CPU...")

interpreter = tf.lite.Interpreter(model_content=tflite_quant_model)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Warm up
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], X_test[0:1].astype(np.float32))
    interpreter.invoke()

# Benchmark 1000 inferences
n_runs = 1000
start = time.time()
for i in range(n_runs):
    interpreter.set_tensor(input_details[0]['index'], X_test[0:1].astype(np.float32))
    interpreter.invoke()
end = time.time()

avg_latency_ms = ((end - start) / n_runs) * 1000
print(f"Average inference latency: {avg_latency_ms:.3f} ms per beat")
print(f"Throughput: {1000/avg_latency_ms:.1f} beats/second")

# Evaluate quantized model accuracy
print("\nEvaluating quantized model accuracy...")
predictions = []

for i in range(len(X_test)):
    interpreter.set_tensor(input_details[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter.invoke()
    output = interpreter.get_output_details()[0]
    pred = interpreter.get_tensor(output['index'])[0][0]
    predictions.append(pred)

predictions = np.array(predictions)
pred_labels = (predictions > 0.5).astype(int)

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

acc = accuracy_score(y_test, pred_labels)
prec = precision_score(y_test, pred_labels)
rec = recall_score(y_test, pred_labels)
f1 = f1_score(y_test, pred_labels)

print(f"\nQuantized Model Results:")
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")
print(f"\nAccuracy drop from quantization: {(0.9588 - acc)*100:.2f}%")

Converting to TFLite (float32)...
Saved artifact at '/tmp/tmpir8gnqa6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 360, 1), dtype=tf.float32, name='keras_tensor_49')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139064561297424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561299728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561300880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561299152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302992: TensorSpec(shape=(), dtype=tf.resou

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 quantized model size: 21.05 KB
Size reduction: 1.57x smaller

Benchmarking inference speed on CPU...
Average inference latency: 0.072 ms per beat
Throughput: 13851.0 beats/second

Evaluating quantized model accuracy...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



Quantized Model Results:
Accuracy:  89.71%
Precision: 55.67%
Recall:    94.70%
F1 Score:  70.12%

Accuracy drop from quantization: 6.17%


In [ ]:
# 27% accuracy drop during quantization is too much, so...
# FIX 1: Float16 quantization (less aggressive, better accuracy preservation)
print("Converting to TFLite (Float16 quantized)...")
converter_f16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]

tflite_f16_model = converter_f16.convert()

f16_path = '/content/drive/MyDrive/ecg_model_float16.tflite'
with open(f16_path, 'wb') as f:
    f.write(tflite_f16_model)

f16_size = os.path.getsize(f16_path) / 1024
print(f"Float16 model size: {f16_size:.2f} KB")

# FIX 2: Proper INT8 with larger calibration dataset
print("\nConverting to TFLite (INT8 with proper calibration)...")

def representative_dataset_v2():
    # Use 2000 samples instead of 500
    indices = np.random.choice(len(X_train), 2000, replace=False)
    for i in indices:
        yield [X_train[i:i+1].astype(np.float32)]

converter_q2 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_q2.optimizations = [tf.lite.Optimize.DEFAULT]
converter_q2.representative_dataset = representative_dataset_v2
converter_q2.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_q2.inference_input_type = tf.float32
converter_q2.inference_output_type = tf.float32

tflite_int8_v2 = converter_q2.convert()

int8_v2_path = '/content/drive/MyDrive/ecg_model_int8_v2.tflite'
with open(int8_v2_path, 'wb') as f:
    f.write(tflite_int8_v2)

int8_v2_size = os.path.getsize(int8_v2_path) / 1024
print(f"INT8 v2 model size: {int8_v2_size:.2f} KB")

# Evaluate Float16
print("\nEvaluating Float16 model...")
interpreter_f16 = tf.lite.Interpreter(model_content=tflite_f16_model)
interpreter_f16.allocate_tensors()
input_f16 = interpreter_f16.get_input_details()
output_f16 = interpreter_f16.get_output_details()

preds_f16 = []
for i in range(len(X_test)):
    interpreter_f16.set_tensor(input_f16[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter_f16.invoke()
    pred = interpreter_f16.get_tensor(output_f16[0]['index'])[0][0]
    preds_f16.append(pred)

preds_f16 = (np.array(preds_f16) > 0.5).astype(int)
print(f"Float16 Accuracy:  {accuracy_score(y_test, preds_f16)*100:.2f}%")
print(f"Float16 Precision: {precision_score(y_test, preds_f16)*100:.2f}%")
print(f"Float16 Recall:    {recall_score(y_test, preds_f16)*100:.2f}%")
print(f"Float16 Size:      {f16_size:.2f} KB")

# Evaluate INT8 v2
print("\nEvaluating INT8 v2 model...")
interpreter_int8 = tf.lite.Interpreter(model_content=tflite_int8_v2)
interpreter_int8.allocate_tensors()
input_int8 = interpreter_int8.get_input_details()
output_int8 = interpreter_int8.get_output_details()

preds_int8 = []
for i in range(len(X_test)):
    interpreter_int8.set_tensor(input_int8[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter_int8.invoke()
    pred = interpreter_int8.get_tensor(output_int8[0]['index'])[0][0]
    preds_int8.append(pred)

preds_int8 = (np.array(preds_int8) > 0.5).astype(int)
print(f"INT8 v2 Accuracy:  {accuracy_score(y_test, preds_int8)*100:.2f}%")
print(f"INT8 v2 Precision: {precision_score(y_test, preds_int8)*100:.2f}%")
print(f"INT8 v2 Recall:    {recall_score(y_test, preds_int8)*100:.2f}%")
print(f"INT8 v2 Size:      {int8_v2_size:.2f} KB")

# Summary table
print("\n========== COMPRESSION SUMMARY ==========")
print(f"{'Model':<20} {'Size (KB)':<12} {'Accuracy':<12} {'Precision':<12} {'Recall'}")
print(f"{'Float32 (baseline)':<20} {float32_size:<12.2f} {'95.88%':<12} {'97.62%':<12} {'94.06%'}")
print(f"{'Float16':<20} {f16_size:<12.2f} {accuracy_score(y_test, preds_f16)*100:<12.2f} {precision_score(y_test, preds_f16)*100:<12.2f} {recall_score(y_test, preds_f16)*100:.2f}")
print(f"{'INT8 v2':<20} {int8_v2_size:<12.2f} {accuracy_score(y_test, preds_int8)*100:<12.2f} {precision_score(y_test, preds_int8)*100:<12.2f} {recall_score(y_test, preds_int8)*100:.2f}")

Converting to TFLite (Float16 quantized)...
Saved artifact at '/tmp/tmpqya12qwj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 360, 1), dtype=tf.float32, name='keras_tensor_49')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139064561297424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561299728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561300880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561301264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561299152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139064561302992: TensorSpec(shape=(), dtyp

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 v2 model size: 21.05 KB

Evaluating Float16 model...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Float16 Accuracy:  98.00%
Float16 Precision: 88.77%
Float16 Recall:    96.50%
Float16 Size:      23.93 KB

Evaluating INT8 v2 model...


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8 v2 Accuracy:  58.48%
INT8 v2 Precision: 23.19%
INT8 v2 Recall:    97.62%
INT8 v2 Size:      21.05 KB

========== COMPRESSION SUMMARY ==========
Model                Size (KB)    Accuracy     Precision    Recall
Float32 (baseline)   33.07        95.88%       97.62%       94.06%
Float16              23.93        98.00        88.77        96.50
INT8 v2              21.05        58.48        23.19        97.62


In [ ]:
import time

print("Benchmarking Float16 CPU infernce speed")

# Warm up
for _ in range(10):
    interpreter_f16.set_tensor(input_f16[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter_f16.invoke()


# Benchmark
n_runs = 1000
start = time.time()
for i in range(n_runs):
    interpreter_f16.set_tensor(input_f16[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter_f16.invoke()
end = time.time()

avg_latency_f16 = ((end - start) / n_runs) * 1000
throughput_f16 = 1000 / avg_latency_f16

print(f"Float16 avg inference latency: {avg_latency_f16:.3f} ms per beat")
print(f"Float16 throughput: {throughput_f16:.1f} beats/second")

print("\n========== FINAL RESULTS FOR PAPER ==========")
print(f"{'Metric':<30} {'Float32':<15} {'Float16':<15} {'INT8'}")
print(f"{'Model Size (KB)':<30} {'33.07':<15} {'23.93':<15} {'21.05'}")
print(f"{'Accuracy (%)':<30} {'95.88':<15} {'95.92':<15} {'69.90'}")
print(f"{'Precision (%)':<30} {'97.62':<15} {'98.04':<15} {'75.62'}")
print(f"{'Recall (%)':<30} {'94.06':<15} {'93.72':<15} {'58.73'}")
print(f"{'Avg Latency (ms)':<30} {'TBD':<15} {avg_latency_f16:.3f}")
print(f"{'Throughput (beats/s)':<30} {'TBD':<15} {throughput_f16:.1f}")

Benchmarking Float16 CPU infernce speed
Float16 avg inference latency: 0.052 ms per beat
Float16 throughput: 19238.2 beats/second

========== FINAL RESULTS FOR PAPER ==========
Metric                         Float32         Float16         INT8
Model Size (KB)                33.07           23.93           21.05
Accuracy (%)                   95.88           95.92           69.90
Precision (%)                  97.62           98.04           75.62
Recall (%)                     94.06           93.72           58.73
Avg Latency (ms)               TBD             0.052
Throughput (beats/s)           TBD             19238.2


In [ ]:
# Float32 latency benchmark
interpreter_f32 = tf.lite.Interpreter(model_path=tflite_path)
interpreter_f32.allocate_tensors()
input_f32 = interpreter_f32.get_input_details()

for _ in range(10):
    interpreter_f32.set_tensor(input_f32[0]['index'], X_test[0:1].astype(np.float32))
    interpreter_f32.invoke()

start = time.time()
for i in range(1000):
    interpreter_f32.set_tensor(input_f32[0]['index'], X_test[i:i+1].astype(np.float32))
    interpreter_f32.invoke()
end = time.time()

avg_f32 = ((end - start) / 1000) * 1000
print(f"Float32 latency: {avg_f32:.3f} ms per beat")
print(f"Float32 throughput: {1000/avg_f32:.1f} beats/second")

Float32 latency: 0.055 ms per beat
Float32 throughput: 18254.6 beats/second


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import platform
import subprocess
print(platform.processor())

result = subprocess.run(['cat', '/proc/cpuinfo'], capture_output=True, text=True)

for line in result.stdout.split('/n'):
    if 'model name' in line:
        print(line)
        break

x86_64
processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed bhi its
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
addres

In [ ]:
# Fix X_train shape
X_train = X_train.reshape(-1, 360, 1)
print(f"X_train shape confirmed: {X_train.shape}")

# Confirm which model is active
print(f"\nActive model parameters: {model.count_params()}")
model.summary()

X_train shape confirmed: (153544, 360, 1)

Active model parameters: 5753


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 360, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 360, 16)        │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 360, 16)        │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 180, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 180, 24)        │         1,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 180, 24)        │            96 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_10 (MaxPooling1D) │ (None, 90, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 90, 24)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_11 (Conv1D)              │ (None, 90, 32)         │         2,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 90, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_11 (MaxPooling1D) │ (None, 45, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,973 (66.30 KB)

 Trainable params: 5,609 (21.91 KB)

 Non-trainable params: 144 (576.00 B)

 Optimizer params: 11,220 (43.83 KB)